# Visión por computadora con modelos abiertos (Hugging Face)
**Cómputo en la Nube para ML/AI · Visión por computadora**

Rekognition es un servicio administrado: mandas una imagen y AWS se encarga del modelo. Aquí hacemos tareas parecidas con **modelos abiertos** que descargamos y corremos nosotros mismos en la notebook instance.

| Parte | Tarea | Modelo | Equivalente en Rekognition |
|---|---|---|---|
| 1 | Clasificación de imágenes | ViT (Google) | `detect_labels` |
| 2 | Detección de objetos | DETR (Meta) | `detect_labels` con cajas |
| 3 | Detección **zero-shot**: tú eliges qué buscar | OWL-ViT (Google) | `detect_protective_equipment`, pero con cualquier objeto |
| 4 | Clasificación zero-shot con texto libre | CLIP (OpenAI) | — |
| 5 | Detección de rostros | Viola–Jones (OpenCV) | `detect_faces` |
| 6 | Buscador de fotos con texto libre | CLIP | — |
| 7 | Descripción automática de imágenes | BLIP (Salesforce) | — |
| 8 | ¿Servicio administrado o modelo abierto? | — | — |

**Kernel recomendado:** `conda_pytorch_p310`. La primera vez cada modelo se descarga de Hugging Face (de 150 MB a 1 GB cada uno), así que la notebook instance necesita salida a internet y unos 3 GB libres.

## 0 · Configuración

In [ ]:
import importlib, subprocess, sys
for paquete, modulo in [("transformers", "transformers"), ("timm", "timm"), ("scikit-image", "skimage"), ("matplotlib", "matplotlib"), ("scikit-learn", "sklearn"), ("requests", "requests")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=False)

# OpenCV 5 eliminó el detector clásico de rostros (CascadeClassifier); fijamos la versión 4
try:
    import cv2
    falta_cascada = not hasattr(cv2, "CascadeClassifier")
except ImportError:
    falta_cascada = True
if falta_cascada:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless<5"], check=False)
    print("Instalé OpenCV 4. Si la Parte 5 marca error, reinicia el kernel (Kernel → Restart) y vuelve a correr desde aquí.")
print("Dependencias listas ✔")

In [ ]:
import io, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import torch
from transformers import pipeline

DISPOSITIVO = 0 if torch.cuda.is_available() else -1
print("GPU disponible:", torch.cuda.is_available(), "· (en CPU cada modelo tarda unos segundos por imagen)")

CARPETA = Path("imagenes"); CARPETA.mkdir(exist_ok=True)

IMAGENES = {
    "escena":  "sklearn:china.jpg",     # templo (viene con scikit-learn)
    "flor":    "sklearn:flower.jpg",
    "rostro":  "skimage:astronaut",     # foto oficial de NASA de Eileen Collins (dominio público)
    "gato":    "skimage:chelsea",
    "cafe":    "skimage:coffee",
    "cohete":  "skimage:rocket",
}

def cargar(fuente, lado_max=1024):
    if fuente.startswith("sklearn:"):
        from sklearn.datasets import load_sample_image
        img = Image.fromarray(load_sample_image(fuente.split(":", 1)[1]))
    elif fuente.startswith("skimage:"):
        from skimage import data
        img = Image.fromarray(getattr(data, fuente.split(":", 1)[1])())
    elif fuente.startswith("http"):
        import requests
        img = Image.open(io.BytesIO(requests.get(fuente, timeout=30).content))
    else:
        img = Image.open(fuente)
    img = img.convert("RGB"); img.thumbnail((lado_max, lado_max))
    return img

def caja_px(ax, b, color="#FFB020", texto=None):
    # Caja en píxeles, formato de Hugging Face: xmin, ymin, xmax, ymax
    ax.add_patch(patches.Rectangle((b["xmin"], b["ymin"]), b["xmax"] - b["xmin"], b["ymax"] - b["ymin"],
                                   fill=False, lw=2.5, ec=color))
    if texto:
        ax.text(b["xmin"], max(b["ymin"] - 4, 0), texto, fontsize=9, va="bottom",
                bbox=dict(facecolor=color, alpha=0.9, pad=2, lw=0))

def mostrar(img, titulo="", tam=(9, 7)):
    fig, ax = plt.subplots(figsize=tam); ax.imshow(img); ax.axis("off"); ax.set_title(titulo); return ax

def cronometro(f, *a, **k):
    t = time.time(); r = f(*a, **k); return r, time.time() - t

print("Listo ✔")

---
## 1 · Clasificación de imágenes (ViT)

Un **Vision Transformer** entrenado con ImageNet: 1,000 categorías fijas. Le das una imagen y te dice qué es, con una probabilidad.

In [ ]:
clasificador = pipeline("image-classification", model="google/vit-base-patch16-224", device=DISPOSITIVO)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, clave in zip(axs, ["gato", "cafe", "cohete"]):
    img = cargar(IMAGENES[clave])
    res, seg = cronometro(clasificador, img, top_k=5)
    ax.barh([r["label"][:28] for r in res][::-1], [r["score"] for r in res][::-1], color="#6C4FF6")
    ax.set_title(f"{clave} · {seg:.1f} s"); ax.set_xlim(0, 1)
plt.tight_layout(); plt.show()

💬 Solo conoce las 1,000 clases con las que se entrenó. ¿Qué responde si le das algo que no está en esa lista?

---
## 2 · Detección de objetos (DETR)

**DETR** (Meta, 2020) encuentra objetos y su ubicación. Conoce las 80 categorías del dataset COCO: personas, autos, tazas, perros…

In [ ]:
detector = pipeline("object-detection", model="facebook/detr-resnet-50", device=DISPOSITIVO)

for clave in ["cafe", "rostro", "escena"]:
    img = cargar(IMAGENES[clave])
    res, seg = cronometro(detector, img, threshold=0.7)
    ax = mostrar(img, f"DETR · {clave} · {len(res)} objetos · {seg:.1f} s")
    for r in res:
        caja_px(ax, r["box"], texto=f'{r["label"]} {r["score"]:.0%}')
    plt.show()

💬 Cambia `threshold` a 0.3 y a 0.95 y mira los resultados.

---
## 3 · Detección zero-shot: tú decides qué buscar (OWL-ViT)

Aquí está la diferencia más interesante con un modelo cerrado: **OWL-ViT** detecta objetos descritos con texto, aunque **nunca lo hayan entrenado con esa clase**. Es como el detector de equipo de protección de Rekognition, pero con cualquier objeto que se te ocurra.

In [ ]:
detector_libre = pipeline("zero-shot-object-detection", model="google/owlvit-base-patch32", device=DISPOSITIVO)

BUSCAR = ["a cup", "a spoon", "a saucer", "a human face", "a helmet", "a flag"]   # Intenta cambiar esta lista

for clave in ["cafe", "rostro"]:
    img = cargar(IMAGENES[clave])
    res, seg = cronometro(detector_libre, img, candidate_labels=BUSCAR, threshold=0.1)
    ax = mostrar(img, f"OWL-ViT · {clave} · {seg:.1f} s")
    for r in res:
        caja_px(ax, r["box"], "#14B8A6", f'{r["label"]} {r["score"]:.0%}')
    plt.show()

💬 Prueba la lista en español (`["una taza", "una cuchara"]`). ¿Funciona igual? Estos modelos se entrenaron sobre todo con texto en inglés.

---
## 4 · Clasificación zero-shot con texto libre (CLIP)

**CLIP** (OpenAI, 2021) aprendió con 400 millones de pares imagen–texto de internet. No tiene clases fijas: tú escribes las opciones y elige la que mejor describe la imagen.

In [ ]:
clip_clf = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32", device=DISPOSITIVO)

OPCIONES = ["a photo of an astronaut", "a photo of a doctor", "a photo of a pilot", "a photo of a student"]
img = cargar(IMAGENES["rostro"])
res = clip_clf(img, candidate_labels=OPCIONES)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].imshow(img); axs[0].axis("off")
axs[1].barh([r["label"] for r in res][::-1], [r["score"] for r in res][::-1], color="#6C4FF6"); axs[1].set_xlim(0, 1)
plt.tight_layout(); plt.show()

💬 **Sesgos:** cambia las opciones por profesiones o adjetivos sobre personas. Un modelo que aprendió de internet también aprendió sus estereotipos. Es el mismo tema de *Gender Shades* que vimos en la presentación.

---
## 5 · Detección de rostros con OpenCV (Viola–Jones, 2001)

Un detector clásico, anterior al deep learning, que viene incluido en OpenCV y no descarga nada. Encuentra rostros, pero **no** estima edad, emociones ni identidad.

In [ ]:
import cv2

cascada = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

for clave in ["rostro"] + (["grupo"] if "grupo" in IMAGENES else []):
    img = cargar(IMAGENES[clave])
    gris = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    (caras), seg = cronometro(cascada.detectMultiScale, gris, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    ax = mostrar(img, f"OpenCV · {clave} · {len(caras)} rostro(s) · {seg*1000:.0f} ms")
    for (x, y, w, h) in caras:
        caja_px(ax, {"xmin": x, "ymin": y, "xmax": x + w, "ymax": y + h}, "#FF6B6B", "rostro")
    plt.show()

💬 Es muy rápido, pero falla con rostros de perfil, inclinados o con poca luz. Rekognition usa redes profundas y además estima atributos. ¿Cuándo bastaría con este detector?

---
## 6 · Buscador de fotos con texto libre (CLIP)

En el notebook de Rekognition el buscador solo encontraba las etiquetas que el servicio conoce. Con CLIP puedes buscar con **cualquier frase**, porque imágenes y textos se convierten en vectores del mismo espacio y se comparan con similitud coseno.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

fuentes = list(IMAGENES.values()) + [str(p) for p in CARPETA.glob("*.jpg")]
fotos = [cargar(f, 512) for f in fuentes]

with torch.no_grad():
    emb_img = clip.get_image_features(**proc(images=fotos, return_tensors="pt"))
emb_img = emb_img.pooler_output
print(f"{len(fotos)} fotos convertidas en vectores de {emb_img.shape[1]} dimensiones")

def buscar(frase, k=3):
    with torch.no_grad():
        emb_txt = clip.get_text_features(**proc(text=[frase], return_tensors="pt", padding=True))
    emb_txt = emb_txt.pooler_output
    sim = (emb_img @ emb_txt.T).squeeze(1).cpu().numpy()
    top = np.argsort(-sim)[:k]
    fig, axs = plt.subplots(1, len(top), figsize=(4 * len(top), 4), squeeze=False)
    for ax, i in zip(axs[0], top):
        ax.imshow(fotos[i]); ax.axis("off"); ax.set_title(f"{Path(fuentes[i]).name}\nsimilitud {sim[i]:.2f}")
    plt.suptitle(f'"{frase}"'); plt.show()

buscar("something you can drink")

Prueba `buscar("a place to pray")`, `buscar("a cute animal")` o `buscar("space exploration")`. Ninguna de esas frases es una etiqueta: el modelo entiende la **relación** entre texto e imagen.

---
## 7 · Descripción automática de imágenes (BLIP) · opcional

Genera una frase que describe la imagen. Es el modelo más pesado del notebook (≈1 GB).

In [ ]:
describir = pipeline("image-to-text", model="Salesforce/blip-image-captioning-base", device=DISPOSITIVO)

fig, axs = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, clave in zip(axs, ["cafe", "escena", "gato"]):
    img = cargar(IMAGENES[clave])
    res, seg = cronometro(describir, img, text="")
    if isinstance(res, list):
        texto_generado = res[0]["generated_text"]
    elif isinstance(res, dict):
        texto_generado = res["generated_text"]
    else:
        texto_generado = str(res)
        
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f'"{texto_generado}"\n({seg:.1f} s)', fontsize=10)
plt.tight_layout(); plt.show()

---
## 8 · ¿Servicio administrado o modelo abierto?

| Criterio | Amazon Rekognition | Modelo abierto (Hugging Face) |
|---|---|---|
| **Arranque** | Una llamada a la API | Descargar el modelo, instalar librerías, tener cómputo |
| **Infraestructura** | La administra AWS | La administras tú: notebook, endpoint, GPU |
| **Costo** | Por imagen procesada | Por hora de la máquina donde corre, la uses o no |
| **Flexibilidad** | Solo lo que ofrece el catálogo | Cualquier modelo; lo puedes ajustar con tus datos |
| **Privacidad** | La imagen viaja al servicio | La imagen no sale de tu infraestructura |
| **Mantenimiento** | AWS actualiza el modelo | Tú decides cuándo y cómo actualizar |
| **Rendimiento en CPU** | No aplica: corre en la infraestructura de AWS | Mira los tiempos que midieron arriba |